In [1]:
from data.cleaning import read_csv
from core import Config
import pandas as pd

config = Config()

static_df: pd.DataFrame = read_csv(
    config.selected_dir / "selected_static.csv",
    config.selected_dir / "selected_static_dtypes.csv",
    [0]
)

historic_df: pd.DataFrame = read_csv(
    config.selected_dir / "selected_historic.csv",
    config.selected_dir / "selected_historic_dtypes.csv",
    [0]
)

# Impute static dataframe

In [2]:
from data.constants.constants_hq import HQ
from data.imputation import fill_na_by_modes, fill_na_by_median

static_df = fill_na_by_modes(static_df)
static_df = fill_na_by_median(static_df)
static_df['TR.HeadquartersCity'] = static_df['TR.HeadquartersCity'].fillna(HQ)

# Filter and convert historic dataframe

In [3]:
# Detect columns with only boolean-like strings and convert dtype to boolean
def is_bool_string_col(series):
    bool_strings = [{'true', 'false'}, {'True', 'False'}, {'yes', 'no'}, {'Yes', 'No'}, {'1', '0'}]
    values = set(series.dropna().unique())
    return any(values <= s for s in bool_strings)

def transform_bool_strings(dataframe: pd.DataFrame) -> pd.DataFrame:
    df = dataframe.copy()
    for col in df.select_dtypes(include=['object', 'string']):
        if is_bool_string_col(df[col]):
            df[col] = df[col].replace(
                {'true': True, 'True': True, 'yes': True, 'Yes': True, '1': True,
                'false': False, 'False': False, 'no': False, 'No': False, '0': False}).astype('boolean')
            df[col] = df[col].astype('boolean')
    return df

# remove columns with only one unique (non‑NaN) value
def remove_columns_with_only_one_unique_value(dataframe: pd.DataFrame) -> pd.DataFrame:
    df = dataframe.copy()
    nunique = df.nunique(dropna=True)
    cols_to_drop = nunique[nunique == 1].index
    return df.drop(columns=cols_to_drop)

historic_df = transform_bool_strings(historic_df)
historic_df = remove_columns_with_only_one_unique_value(historic_df)

# Merge static and Historic Frame

In [2]:
# Join static and historic dataframe to have GICS Sector Codes in the dataframe
all_df = historic_df.merge(
    static_df,
    on='Instrument',
    how='left'
)

In [5]:
# OPTIONAL: filter rows with a lot of nan values. 1_788 rows with over 100 nan values
# all_df = all_df.drop(all_df.loc[all_df.isna().sum(axis=1) > 100, all_df.columns].index)

# Compute Median and Mode

In [6]:
# Computing the median not only per sector but also aligning them per year. So every median is computed from values of the same year.
# Only if there is not enough data, then compute only per sector.
from data.imputation import calculate_median, calculate_mode

all_df2 = all_df.dropna(subset=["TR.UpstreamScope3PurchasedGoodsAndServices"], inplace=True)
all_df2 = calculate_median(all_df)
all_df2 = calculate_mode(all_df2)

if all_df2.isna().sum().sum() > 0:
    print(f"Imputation: {all_df2.isna().sum().sum()} values are not Zero!!!")

In [3]:
all_df['Instrument'] = all_df['Instrument'].astype('category')
all_df.to_csv(config.data_dir / "datasets" / "median" / "median_imputed.csv")
dtypes: pd.DataFrame = all_df.dtypes.to_frame('dtypes').reset_index()
dtypes.to_csv(config.data_dir / "datasets" / "median" / "median_imputed_dtypes.csv")